In [1]:
from typing import List, Dict, Tuple, Union, Literal, Optional

import pandas as pd
from openai import AsyncOpenAI, OpenAI
from openai.lib._pydantic import to_strict_json_schema
from PIL import Image
import base64
import requests
from pydantic import BaseModel

In [2]:
ROOMS =  ["Kitchen", "Bathroom", "Garden", "Office", "Bedroom", "Hallway"]
CHARS = ["Daniel", "Mary", "Michael", "Sandra", "John"]
NOBODY = "Nobody"

In [3]:
class RoomAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: Literal[*ROOMS]


class NumberAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: int = 0


class PersonAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: Literal[*CHARS, NOBODY] = NOBODY

In [16]:
answer = """<think>
In step 1, the 'step_id':1 dictionary only contains rooms and the agents present in the 'Office', 'Bedroom', and 'Hallway'. No rooms are listed as empty, but based on the information given, we can deduce that the Kitchen, Garden, and potentially other unlisted rooms may have been empty.
</think>



<answer>
4
</answer>
"""

In [2]:
import re

In [11]:
soft_re = r"<think>.*?</think>\s*<answer>.*?</answer>"

In [17]:
re.match(soft_re, answer, re.DOTALL)

<re.Match object; span=(0, 328), match="<think>\nIn step 1, the 'step_id':1 dictionary on>

In [2]:
import os
import json
from natsort import natsorted

def concatenate_jsons(directory, output_file):
    # Initialize an empty list to hold all JSON data
    combined_data = []

    # Get a list of all JSON files in the directory and sort them by filename
    json_files = natsorted([f for f in os.listdir(directory) if f.endswith('.json')])

    # Iterate over the sorted JSON files
    for filename in json_files:
        print(filename)
        file_path = os.path.join(directory, filename)
        
        # Open and load the JSON file
        with open(file_path, 'r') as file:
            data = json.load(file)
            
            # If the JSON file contains a list, extend the combined_data list
            if isinstance(data, list):
                combined_data.extend(data)
            else:
                # If it's a dictionary, append it to the combined_data list
                combined_data.append(data)

    # Write the combined data to a new JSON file
    with open(output_file, 'w') as outfile:
        json.dump(combined_data, outfile)

    print(f"All JSON files have been concatenated into {output_file}")

# Example usage
directory = '/workspace-SR004.nfs2/data/long_vqa_synth/main_1mv/text_description_serialized/'
output_file = '/workspace-SR004.nfs2/data/long_vqa_synth/main_1mv/text_description_serialized.json'
concatenate_jsons(directory, output_file)

len_1_text_description_serialized_questions.json
len_2_text_description_serialized_questions.json
len_4_text_description_serialized_questions.json
len_8_text_description_serialized_questions.json
len_16_text_description_serialized_questions.json
len_32_text_description_serialized_questions.json
len_64_text_description_serialized_questions.json
len_128_text_description_serialized_questions.json
All JSON files have been concatenated into /workspace-SR004.nfs2/data/long_vqa_synth/main_1mv/text_description_serialized.json


In [1]:
from openai import OpenAI, AsyncOpenAI

client = AsyncOpenAI(api_key="YOUR_API_KEY", base_url="http://0.0.0.0:8003/v1")
model_names = await client.models.list()
model_name = model_names.data[0].id
model_name

'Qwen/Qwen2.5-VL-3B-Instruct'

In [4]:
prompt = """
Analyze the image and provide a detailed description for it.
Also include detailed structured description for each person in the frame, including: 
    - physical appearance (clothing, accesories, hairstyle, eye color, big/small parts of face and body, etc.)
    - current actions
    - emotions
    - exact position in the frame
    - bounding box coordinates of <ref>{detailed description of this person and his relative position}</ref>
Additionally, indicate the total number of people, describe any interactions 
between them or with objects/animals, and note any occlusions or overlaps.
Ensure the description is thorough for each individual.
"""

In [4]:
prompt = ""

In [5]:
files = [
    "/workspace-SR004.nfs2/kurkin/long-vqa/data/length_128/sequence_000/step_001.png"
]

In [6]:
import base64
import os
from io import BytesIO
from typing import Union

import requests
from PIL import Image, ImageFile


def encode_image_base64(image: Union[str, Image.Image]) -> str:
    """encode raw date to base64 format."""
    buffered = BytesIO()
    FETCH_TIMEOUT = int(os.environ.get("LMDEPLOY_FETCH_TIMEOUT", 10))
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
    }
    try:
        if isinstance(image, str):
            url_or_path = image
            if url_or_path.startswith("http"):
                response = requests.get(
                    url_or_path, headers=headers, timeout=FETCH_TIMEOUT
                )
                response.raise_for_status()
                buffered.write(response.content)
            elif os.path.exists(url_or_path):
                with open(url_or_path, "rb") as image_file:
                    buffered.write(image_file.read())
        elif isinstance(image, Image.Image):
            image.save(buffered, format="PNG")
    except Exception as error:
        if isinstance(image, str) and len(image) > 100:
            image = image[:100] + " ..."
        print(f"{error}, image={image}, using dummy image")
        # use dummy image
        image = Image.new("RGB", (32, 32))
        image.save(buffered, format="PNG")
    res = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return res

In [71]:
import json

In [73]:
schemas["number"]

{'properties': {'reasoning': {'anyOf': [{'maxLength': 1250, 'type': 'string'},
    {'type': 'null'}],
   'default': None,
   'title': 'Reasoning'},
  'answer': {'default': 0,
   'minimum': 0,
   'title': 'Answer',
   'type': 'integer'}},
 'title': 'NumberAnswer',
 'type': 'object'}

In [72]:
json.dumps(schemas["number"])

'{"properties": {"reasoning": {"anyOf": [{"maxLength": 1250, "type": "string"}, {"type": "null"}], "default": null, "title": "Reasoning"}, "answer": {"default": 0, "minimum": 0, "title": "Answer", "type": "integer"}}, "title": "NumberAnswer", "type": "object"}'

In [70]:
print(response.choices[0].message.content)

To determine how many characters were present in the bedroom when John made his final appearance in the hallway, we need to analyze both images provided and track the movements of each character over time.

1. Initially at Step 1:
    - Kitchen: No one
    - Bathroom: Sandra (blue dot)
    - Garden: Mary & Michael (red and purple dots)
    - Office: John (green dot)
    - Bedroom: Daniel (yellow dot)
    - Hallway: No one

2. At some point before Step 2:
    - Sandra moves from Bathroom to Bedroom.
    - John moves from Office to Hallway.

3. At Step 2:
    - Kitchen: No one
    - Bathroom: No one
    - Garden: Mary & Michael (red and purple dots)
    - Office: No one
    - Bedroom: Sandra & Daniel (blue and yellow dots)
    - Hallway: John (green dot)

From this sequence, we can see that John was last seen in the hallway after moving out of the bedroom. Therefore, during John's final appearance in the hallway, there were two characters in the bedroom: Sandra and Daniel.

Therefore, th

In [17]:
import requests
import torch
from PIL import Image

from transformers import AriaProcessor, AriaForConditionalGeneration


model_id_or_path = "rhymes-ai/Aria"

processor = AriaProcessor.from_pretrained(model_id_or_path)

In [25]:
processor.tokenizer.decode([1970, 93653])

'<|'

In [19]:
processor.tokenizer("<|im_end|>", add_special_tokens=False)

{'input_ids': [1970, 93653, 944, 93421, 1019, 93653, 93519], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [23]:
prefix = ""
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that analyzes image sequences. Answer with one word. Where was Mary last time?",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": prefix
                    + "/home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_001.png",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": prefix
                    + "/home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_001.png",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": prefix
                    + "/home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_002.png",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": prefix
                    + "/home/jovyan/shares/SR004.nfs2/kurkin/omni/long-vqa/data/length_256/sequence_000/step_003.png",
                },
            },
        ],
    }
]

In [24]:
response = await client.chat.completions.create(
    model=model_name, messages=messages, temperature=0.0, top_p=0.8
)
print(response)

ChatCompletion(id='31', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Mary was last in the hallway.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1735411299, model='OpenGVLab/InternVL2_5-78B-MPO', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=7, prompt_tokens=13380, total_tokens=13387, completion_tokens_details=None, prompt_tokens_details=None))


In [25]:
len(response.choices)

1

In [26]:
print(response.choices[0].message.content)

Mary was last in the hallway.


In [27]:
import asyncio

responses = await asyncio.gather(
    *[
        client.chat.completions.create(
            model=model_name, messages=messages, temperature=0.0, top_p=0.8
        )
        for _ in range(4)
    ]
)